# Figure 1, panels b–d — regulators and gene programs during arterial endothelial differentiation

This notebook contains the source-verified computational producers for panels **b–d**. Panel a is omitted because it is an assembled schematic rather than a computational panel.

- **Panel b:** `Pos_vs_Neg_volcano_plotnine_MAGECK_FDR_pathway_colored_v8.py`.
- **Panel c:** attached `validation_boxplots_verified` bundle. The analysis uses `estimated_EC_pct_control70` directly, 120 individual NON-TARGETING/SAFE control guides, 100,000 two-sided pooled guide-label permutations, and BH correction across 297 genes.
- **Panel d:** Sherlock `programDE/notebooks/per_day_umaps/jesse_umap.ipynb`, using the integrated UMAP and cNMF H5MU object.

Machine-specific inputs are read from `config/paths.local.yaml`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import matplotlib.pyplot as plt

from src.cc_perturbseq.figure_notebook import (
    get_path, initialize_figure, load_figure_manifest, panel_directory, sha256_file, write_receipt,
)
from src.cc_perturbseq.figure1_panel_b import make_figure_01b
from src.cc_perturbseq.figure1_panel_c import (
    PANEL_C_EXPECTED_SHA256, PANEL_C_GROUPS, run_figure_01c,
)
from src.cc_perturbseq.figure1_panel_d import make_figure_01d


def save_figure(fig, outdir: Path, stem: str, *, png_dpi: int = 600, pad_inches: float = 0.03):
    outputs = {}
    for suffix in ('pdf', 'svg', 'png'):
        path = outdir / f'{stem}.{suffix}'
        fig.savefig(
            path, dpi=png_dpi if suffix == 'png' else None,
            bbox_inches='tight', pad_inches=pad_inches, facecolor='white',
        )
        outputs[suffix] = str(path)
    return outputs

FIGURE = load_figure_manifest('Figure_01')
FIGURE['panels'] = [panel for panel in FIGURE['panels'] if panel['panel'] in {'b', 'c', 'd'}]
FIGURE['scope'] = 'Panels b–d'
ctx = initialize_figure(FIGURE)
ctx['input_audit']

## Panel b — genome-wide CRISPRi screen

The x axis is the U-test log2 fold change. The y axis uses the directional MAGeCK FDR: `neg|fdr` for negative effects and `pos|fdr` for positive effects. A gene is a MAGeCK hit when either directional FDR is below 0.05.

In [ ]:
mageck_path = get_path(ctx['paths'], 'screen.genomewide_mageck_pos_vs_neg')
utest_path = get_path(ctx['paths'], 'screen.genomewide_utest_pos_vs_neg')
fig_b, source_b, labels_b = make_figure_01b(mageck_path, utest_path)
out_b = panel_directory(ctx, 'b')
outputs_b = save_figure(fig_b, out_b, 'Figure_01b_genomewide_screen_volcano')
source_b.to_csv(out_b / 'Figure_01b_source_data.csv', index=False)
labels_b.to_csv(out_b / 'Figure_01b_label_positions.csv', index=False)
write_receipt(
    ctx, 'b', status='SOURCE_VERIFIED_IMPLEMENTATION',
    inputs={
        'mageck': str(mageck_path), 'mageck_sha256': sha256_file(mageck_path),
        'utest': str(utest_path), 'utest_sha256': sha256_file(utest_path),
    },
    outputs={**outputs_b, 'source_data': str(out_b / 'Figure_01b_source_data.csv')},
    parameters={
        'source_script': 'Pos_vs_Neg_volcano_plotnine_MAGECK_FDR_pathway_colored_v8.py',
        'x_axis': 'U-test log2FC', 'y_axis': 'directional MAGeCK FDR',
        'fdr_cutoff': 0.05, 'vertical_reference': 0.0,
    },
)
plt.show()

## Panel c — validation-screen guide-level Day 3 EC fractions

Set `screen.validation_guide_ec_percent` in `config/paths.local.yaml` to the full verified `guide_EC_percent_control70.csv`. The notebook checks its SHA-256. The dashed line is the **mean** of the 120 control-guide estimates.

In [ ]:
validation_path = get_path(ctx['paths'], 'screen.validation_guide_ec_percent')
out_c = panel_directory(ctx, 'c')
fig_c, stats_c, selected_stats_c, outputs_c, validation_hash = run_figure_01c(
    validation_path, out_c, verify_hash=True, permutations=100_000, seed=20260908,
)
write_receipt(
    ctx, 'c', status='SOURCE_VERIFIED_IMPLEMENTATION',
    inputs={'validation_guide_table': str(validation_path), 'sha256': validation_hash},
    outputs={
        **outputs_c,
        'all_gene_statistics': str(out_c / 'permutation_statistics_all_297_genes.csv'),
        'plotted_gene_statistics': str(out_c / 'permutation_statistics_plotted_genes.csv'),
    },
    parameters={
        'expected_input_sha256': PANEL_C_EXPECTED_SHA256,
        'value_column': 'estimated_EC_pct_control70',
        'n_controls': 120, 'n_genes_BH': 297, 'n_permutations': 100000,
        'seed': 20260908, 'pvalue_definition': 'b/B',
        'weighting': 'one unweighted observation per guide',
        'control_reference': 'arithmetic mean across NON-TARGETING and SAFE guides',
        'plotted_groups': PANEL_C_GROUPS,
    },
)
selected_stats_c[['gene', 'n_targeting_guides', 'mean_target_EC_pct', 'FDR_BH', 'significance']]

## Panel d — integrated UMAP colored by dominant selected cNMF program

This is a lean port of `per_day_umaps/jesse_umap.ipynb`. It transfers `X_umap` from the integrated AnnData object after exact cell-ID verification, then assigns each cell to the highest-scoring program listed in `program_order.txt`. Cells with a maximum selected-program score of 0.02 or less are gray.

In [ ]:
h5mu_path = get_path(ctx['paths'], 'ccperturbseq.cnmf_h5mu')
adata_path = get_path(ctx['paths'], 'ccperturbseq.integrated_anndata')
metadata_path = get_path(ctx['paths'], 'ccperturbseq.figure1d_program_metadata')
order_path = get_path(ctx['paths'], 'ccperturbseq.figure1d_program_order')
fig_d, source_d, programs_d, threshold_d = make_figure_01d(
    h5mu_path, adata_path, metadata_path, order_path,
)
out_d = panel_directory(ctx, 'd')
outputs_d = save_figure(
    fig_d, out_d, 'Figure_01d_integrated_program_atlas', png_dpi=300, pad_inches=0.02,
)
source_d.to_csv(
    out_d / 'Figure_01d_cell_program_assignment.csv.gz',
    index=False, compression='gzip',
)
write_receipt(
    ctx, 'd', status='SOURCE_VERIFIED_IMPLEMENTATION',
    inputs={
        'h5mu': str(h5mu_path), 'integrated_anndata': str(adata_path),
        'program_metadata': str(metadata_path), 'program_order': str(order_path),
    },
    outputs={
        **outputs_d,
        'cell_program_assignment': str(out_d / 'Figure_01d_cell_program_assignment.csv.gz'),
    },
    parameters={
        'source_notebook': 'programDE/notebooks/per_day_umaps/jesse_umap.ipynb',
        'programs': programs_d, 'dominant_program_threshold': threshold_d,
        'umap_source': 'integrated AnnData X_umap',
        'program_score_source': 'cNMF modality in cNMF_50_0_2.h5mu',
    },
)
source_d[['dominant_selected_program', 'above_0.02_threshold']].value_counts().sort_index()